# Taller Mutual Information

En este notebook se resuelven los dos desafíos del notebook `mutual_information.ipynb`:
- **Desafío 1**: Métricas de Mutual Information y Entropy para el dataset `encuesta.xls`
- **Desafío 2**: Métricas de Mutual Information y Entropy para el dataset `demanda_laboral.xlsx`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression
from itertools import combinations

plt.style.use("seaborn-v0_8-whitegrid")

## Funciones auxiliares

Definimos funciones para calcular la entropía de cada columna y la información mutua entre todos los pares de atributos.

In [ ]:
def calcular_entropia(df):
    """Calcula la entropia de Shannon para cada columna del dataframe."""
    resultados = {}
    for col in df.columns:
        valores = df[col].dropna()
        counts = valores.value_counts(normalize=True)
        H = entropy(counts, base=2)
        resultados[col] = H
    return pd.Series(resultados, name="Entropia").sort_values(ascending=False)


def calcular_mi_pairwise(df):
    """Calcula Mutual Information entre todos los pares de columnas."""
    cols = df.columns.tolist()
    mi_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for c1, c2 in combinations(cols, 2):
        vals = df[[c1, c2]].dropna()
        # Discretizar valores continuos en bins para mutual_info_score
        v1 = pd.cut(vals[c1], bins=20, labels=False) if vals[c1].nunique() > 20 else vals[c1]
        v2 = pd.cut(vals[c2], bins=20, labels=False) if vals[c2].nunique() > 20 else vals[c2]
        mi = mutual_info_score(v1, v2)
        mi_matrix.loc[c1, c2] = mi
        mi_matrix.loc[c2, c1] = mi
    for c in cols:
        vals = df[c].dropna()
        v = pd.cut(vals, bins=20, labels=False) if vals.nunique() > 20 else vals
        mi_matrix.loc[c, c] = mutual_info_score(v, v)
    return mi_matrix

---
# Desafío 1: Dataset `encuesta.xls`

Importamos el dataset de la encuesta y encontramos las métricas de Mutual Information y Entropy para todos los atributos.

In [ ]:
df_encuesta = pd.read_excel("encuesta.xls")
df_encuesta.head(10)

In [ ]:
df_encuesta.info()

In [ ]:
df_encuesta.describe()

## Entropía de cada atributo - Encuesta

La entropía mide la incertidumbre o cantidad de información contenida en una variable. Una entropía alta indica mayor variabilidad en los datos.

In [ ]:
entropia_encuesta = calcular_entropia(df_encuesta)
print("Entropia de cada atributo (bits):\n")
print(entropia_encuesta.to_string())

In [ ]:
plt.figure(figsize=(10, 5))
entropia_encuesta.sort_values(ascending=True).plot(kind="barh", color="steelblue")
plt.title("Entropía por atributo - Encuesta")
plt.xlabel("Entropía (bits)")
plt.tight_layout()
plt.show()

## Mutual Information entre pares de atributos - Encuesta

La información mutua cuantifica cuánta información comparten dos variables. Un valor alto indica que conocer una variable reduce la incertidumbre sobre la otra.

In [ ]:
mi_encuesta = calcular_mi_pairwise(df_encuesta)
print("Matriz de Mutual Information:\n")
mi_encuesta

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(mi_encuesta.astype(float), annot=True, fmt=".3f", cmap="YlOrRd", linewidths=0.5)
plt.title("Mutual Information - Encuesta")
plt.tight_layout()
plt.show()

## MI Scores usando cada atributo como variable objetivo

Utilizamos `mutual_info_regression` de scikit-learn para calcular los MI scores tomando cada atributo como objetivo.

In [ ]:
def mi_scores_for_target(df, target_col):
    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()
    for col in X.select_dtypes("object").columns:
        X[col], _ = X[col].factorize()
    discrete_features = X.dtypes == int
    mi_scores = mutual_info_regression(X, y, discrete_features=discrete_features, random_state=42)
    return pd.Series(mi_scores, name="MI Scores", index=X.columns).sort_values(ascending=False)

print("MI Scores usando 'I. Mensual' como variable objetivo:\n")
mi_target_encuesta = mi_scores_for_target(df_encuesta, "I. Mensual")
print(mi_target_encuesta.to_string())

In [ ]:
plt.figure(dpi=100, figsize=(8, 5))
mi_target_encuesta.sort_values(ascending=True).plot(kind="barh", color="teal")
plt.title("MI Scores respecto a 'I. Mensual' - Encuesta")
plt.xlabel("MI Score")
plt.tight_layout()
plt.show()

## Conclusiones - Desafío 1

**¿Que puede concluir del proceso y de la información compartida?**

Se puede concluir que los atributos como G. Alimentacion, Consumo Energia y Consumo de Agua comparten una cantidad considerable de informacion mutua con el Ingreso Mensual, lo cual tiene sentido porque a mayor ingreso mayor capacidad de gasto en estos rubros. La entropia de las variables continuas como I. Mensual y G. Alimentacion es alta, lo que indica que tienen una gran variabilidad y aportan bastante informacion al analisis. En cambio las variables binarias como Casa Propia, Automovil y Telefono tienen entropia mas baja porque solo toman dos valores posibles.

**¿Hay atributos redundantes? ¿Se pueden eliminar atributos?**

Si se observa que G. Alimentacion y Consumo de Agua tienen una informacion mutua muy alta con I. Mensual, lo que sugiere que estas variables son parcialmente redundantes ya que se pueden predecir en gran medida a partir del ingreso mensual. Tambien Consumo Energia muestra una relacion fuerte. En un modelo predictivo se podria considerar eliminar alguna de estas variables redundantes para simplificar el modelo sin perder mucha capacidad predictiva, especialmente si se busca reducir la dimensionalidad del conjunto de datos.

---
# Desafío 2: Dataset `demanda_laboral.xlsx`

Importamos el dataset de demanda laboral y encontramos las métricas de Mutual Information y Entropy para todos los atributos.

Definición de variables:
- **SEXO**: 1: Hombre, 2: Mujer
- **POSGRADO**: 1: Tiene al menos un posgrado, 2: No tiene posgrado
- **NATURALEZA**: Naturaleza de la empresa. 1: Pública, 2: Privada
- **INGRESO**: Ingreso mensual en pesos
- **EDAD**: Edad en años cumplidos
- **TIEMPO**: Años de antigüedad

In [ ]:
df_laboral = pd.read_excel("demanda_laboral.xlsx")
df_laboral.head(10)

In [ ]:
df_laboral.info()

In [ ]:
# Eliminamos la columna 'Rete-salud 12.5%' porque tiene todos los valores NaN
df_laboral = df_laboral.drop(columns=["Rete-salud 12.5%"])
df_laboral.describe()

## Entropía de cada atributo - Demanda Laboral

In [ ]:
entropia_laboral = calcular_entropia(df_laboral)
print("Entropia de cada atributo (bits):\n")
print(entropia_laboral.to_string())

In [ ]:
plt.figure(figsize=(10, 5))
entropia_laboral.sort_values(ascending=True).plot(kind="barh", color="darkorange")
plt.title("Entropía por atributo - Demanda Laboral")
plt.xlabel("Entropía (bits)")
plt.tight_layout()
plt.show()

## Mutual Information entre pares de atributos - Demanda Laboral

In [ ]:
mi_laboral = calcular_mi_pairwise(df_laboral)
print("Matriz de Mutual Information:\n")
mi_laboral

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(mi_laboral.astype(float), annot=True, fmt=".3f", cmap="YlOrRd", linewidths=0.5)
plt.title("Mutual Information - Demanda Laboral")
plt.tight_layout()
plt.show()

## MI Scores usando Ingresos como variable objetivo

In [ ]:
print("MI Scores usando 'Ingresos' como variable objetivo:\n")
mi_target_laboral = mi_scores_for_target(df_laboral, "Ingresos")
print(mi_target_laboral.to_string())

In [ ]:
plt.figure(dpi=100, figsize=(8, 5))
mi_target_laboral.sort_values(ascending=True).plot(kind="barh", color="coral")
plt.title("MI Scores respecto a 'Ingresos' - Demanda Laboral")
plt.xlabel("MI Score")
plt.tight_layout()
plt.show()

## Conclusiones - Desafío 2

**¿Que puede concluir del proceso y de la información compartida?**

Se puede concluir que las variables categoricas como Sexo, Posgrado y Naturaleza tienen una entropia baja porque solo toman valores de 1 o 2, lo cual limita su variabilidad. Las variables continuas como Ingresos y Edad presentan una entropia mucho mas alta indicando mayor dispersion en sus valores. Respecto a la informacion mutua, Tiempo y Edad muestran una relacion considerable, lo que es logico porque a mayor edad generalmente mayor tiempo de antiguedad en una empresa.

**¿Hay atributos redundantes?**

Si, se observa que Tiempo y Edad comparten informacion mutua significativa, lo que indica cierta redundancia entre estas variables. Esto tiene sentido intuitivo ya que las personas de mayor edad tienden a tener mas anos de antiguedad laboral. La columna Rete-salud 12.5% fue eliminada porque contenia unicamente valores nulos y no aportaba informacion al analisis. Las variables categoricas como Sexo, Posgrado y Naturaleza aunque tienen baja entropia no son necesariamente redundantes entre si ya que capturan dimensiones diferentes del perfil del encuestado.